# Quantum support vector machines

Regenerate the two original data workflows and train four local classifiers. The two native quantum kernels prepare feature states and sample their all-zero overlap events.

Use the scientific packages installed in the example's local environment.

In [ ]:
from importlib.metadata import version

Import native feature maps, local datasets, classifiers and plots.

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Image, display

import examples.synthesis.algorithms.machine_learning.qsvm.overview.datasets as datasets
import examples.synthesis.algorithms.machine_learning.qsvm.overview.plots as plots
import examples.synthesis.algorithms.machine_learning.qsvm.overview.quantum as quantum
import examples.synthesis.algorithms.machine_learning.qsvm.overview.workflow as workflow
import flagquantum as fq
from examples.synthesis.algorithms.machine_learning.qsvm.overview.main import (
    print_report,
    report_score,
)

figures = Path('notebook-figures')
print('Local scikit-learn:', version('scikit-learn'))

Keep the original synthetic seed and record the local finite kernel budget.

In [ ]:
assert workflow.SHOTS == 2048
print('Synthetic seed:', 0, 'Kernel sample seed:', workflow.SAMPLE_SEED)

Regenerate 40 fresh training, test and prediction rows each.

In [ ]:
synthetic = datasets.synthetic()
assert all(group.points.shape == (40, 2) for group in (synthetic.train, synthetic.test, synthetic.predict))

Check the original normalization on all three synthetic sets.

In [ ]:
assert all(np.all((group.points >= -1) & (group.points <= 1)) for group in (synthetic.train, synthetic.test, synthetic.predict))

Plot training and test labels and the unlabeled prediction inputs.

In [ ]:
plots.synthetic(synthetic, figures)
display(Image(filename=str(figures / 'synthetic-data.png')))

Select the one-qubit Bloch feature map.

In [ ]:
bloch = quantum.FeatureMap.BLOCH
a, b = (0.23, -0.41), (-0.17, 0.37)

Inspect the exact local overlap circuit for two different inputs.

In [ ]:
bloch_program = quantum.build(bloch, a, b)
print(bloch_program.report['resources'])
print(dict(bloch_program.probabilities(fq.run(bloch_program), registers=('q',))))

Train the precomputed-kernel SVC on the full synthetic training set.

In [ ]:
synthetic_bloch = workflow.classify(bloch, synthetic)

Keep actual test labels and the held-out predictions.

In [ ]:
assert synthetic_bloch.test_labels.shape == synthetic_bloch.predictions.shape == (40,)

Report the synthetic classifications and observed accuracies.

In [ ]:
print('Synthetic held-out labels:', tuple(map(int, synthetic.predict.labels)))
report_score('Synthetic Bloch', synthetic_bloch)

Plot the classification of every synthetic prediction row.

In [ ]:
plots.prediction(synthetic.predict, synthetic_bloch, figures)
display(Image(filename=str(figures / 'synthetic-prediction.png')))

Load the original Iris measurements and perform the source preprocessing.

In [ ]:
iris = datasets.iris()

Keep both petal features, both classes and the original angular scale.

In [ ]:
for group in (iris.train, iris.test, iris.predict):
    assert group.points.shape[1] == 2 and set(group.labels) == {0, 1}
    assert np.min(group.points) >= -1e-12 and np.max(group.points) <= 2 * np.pi + 1e-12

Confirm the original stratified 80/10/10 split.

In [ ]:
assert tuple(len(group.labels) for group in (iris.train, iris.test, iris.predict)) == (80, 10, 10)

Plot the complete training and test data after preprocessing.

In [ ]:
plots.iris(iris, figures)
display(Image(filename=str(figures / 'iris-data.png')))

Build the full-connectivity two-layer Z/ZZ map with the original pi offsets.

In [ ]:
pauli = quantum.FeatureMap.PAULI
print('Pauli rotation angles:', quantum.bindings(pauli, a))

Synthesize the two-qubit Pauli overlap.

In [ ]:
pauli_program = quantum.build(pauli, a, b)

Inspect the Pauli resources and verify the complete overlap state.

In [ ]:
print(pauli_program.report['resources'])
left = quantum.independent_circuit(pauli, a)
right = quantum.independent_circuit(pauli, b)
expected = fq.run(left.compose(right.adjoint())).statevector()
actual = fq.run(pauli_program).statevector()
np.testing.assert_allclose(actual.numpy(), expected.numpy(), atol=1e-12, rtol=0)

Fit the sampled Pauli kernel and predict the two Iris holdouts.

In [ ]:
iris_pauli = workflow.classify(pauli, iris)

Build the one-qubit Iris Bloch kernel.

In [ ]:
iris_point = (float(iris.train.points[0, 0]), float(iris.train.points[0, 1]))
print(quantum.build(bloch, iris_point, a).report['resources'])

Fit the sampled Bloch kernel on the same 80 Iris training rows.

In [ ]:
iris_bloch = workflow.classify(bloch, iris)

Report the actual Pauli classifications.

In [ ]:
report_score('Iris Pauli', iris_pauli)

Report the actual Bloch classifications.

In [ ]:
report_score('Iris Bloch', iris_bloch)

Fit the classical RBF comparison and report all observed results.

In [ ]:
iris_rbf = workflow.classical(iris)
result = workflow.Result(synthetic, iris, synthetic_bloch, iris_pauli, iris_bloch, iris_rbf)
print_report(result)